### Imports

In [1]:
import os
import sys
import json
import time
import argparse
import warnings
import xml.etree.ElementTree as ET
import multiprocessing as mp
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
from PIL import Image, ImageDraw

import daisy
import dask
from dask.array import coarsen, mean
from dask.diagnostics import ProgressBar
import zarr
from funlib.persistence import Array, prepare_ds, open_ds
from funlib.geometry import Roi, Coordinate
import tifffile
from tqdm import tqdm
from skimage.measure import label, regionprops
from scipy import ndimage

import rtree
from shapely.geometry import Polygon, box

# for xml > zarr
from shapely.geometry import Polygon, box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from skimage.draw import polygon as draw_polygon

c:\Users\Waluigi\anaconda3\envs\transunet_dataprep\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, os.pardir))
grandparent_dir = os.path.abspath(os.path.join(cwd, os.pardir, os.pardir))

# Import OpenSlide
OPENSLIDE_PATH = os.path.join(grandparent_dir, 'openslide-bin-4.0.0.11-windows-x64\\bin')
print(OPENSLIDE_PATH)
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

c:\Users\Waluigi\Desktop\github_repos\DCIS\openslide-bin-4.0.0.11-windows-x64\bin


### SVS to Zarr

In [ ]:
def opensvs(svs_path, pyramid_level): # from rachel
    """
    open .svs as a dask array and retreive metadata (dimensions + resolution)
    
    INPUTS: 
    - svs_path (str): path to .svs file
    - pyramid_level (int): which pyramid level to open (0 = 40x, 1 = 20x, 2 = 10x, 3 = 5x)
    
    OUTPUTS:
    - dask_array: dask array of the image data for the specified pyramid level
    - x_res: resolution in nm/px in the x dimension
    - y_res: resolution in nm/px in the y dimension
    - units: tuple of units for x and y resolution (should be ("nm", "nm"))
    """
    slide = openslide.OpenSlide(svs_path)
    # grab resolution at micrometeres / px and convert to nm / px
    x_res = float(slide.properties["openslide.mpp-x"]) * 1000
    y_res = float(slide.properties["openslide.mpp-y"]) * 1000
    units = ("nm", "nm")
    # read pyramid level directly — no zarr involved
    with tifffile.TiffFile(svs_path) as tif:
        level = tif.series[0].levels[pyramid_level]
        dask_array = dask.array.from_array(level.asarray(), chunks='auto')

    return dask_array, x_res, y_res, units

def svs_to_zarr(svs_path, zarr_path, offset, axis_names): # from rachel
    """
    convert H&E from .svs to .zarr file

    INPUTS:
    - svs_path (str): path to .svs file 
    - zarr_path (str): path to save .zarr file
    - offset (tuple): offset for the image data (e.g. (0, 0) if no offset)
    - axis_names (tuple): names of the axes (e.g. ("x", "y", "c"))

    OUTPUTS: 
    - saves .zarr file with the image data for each pyramid level (s0 = 40x, s1 = 20x, s2 = 10x, s3 = 5x) and metadata (voxel size, axis names, units)
    """
    # open highest pyramid level (40x)
    dask_array0, x_res, y_res, units = opensvs(svs_path, 0)
    s0_shape = dask_array0.shape
    # units are natively ("nm", "nm") so no need to convert to get voxel size
    
    # convert to integer and calculate for each pyramid level
    voxel_size0 = Coordinate(int(x_res), int(y_res))
    
    # format data as funlib dataset
    raw = prepare_ds(
        zarr_path / "raw" / "s0",
        dask_array0.shape,
        offset,
        voxel_size0,
        axis_names,
        units,
        mode="w",
        dtype=np.uint8,
    )
    
    # storage info
    store_rgb = zarr.open(zarr_path / "raw" / "s0") # s0 = full resolution image at 40x magnification
    dask_array = dask_array0.rechunk(raw.data.chunksize)

    with ProgressBar():
        dask.array.store(dask_array, store_rgb)

    for i in range(1, 4): # iterate over each pyramid level: s1 (20x), s2 (10x), s3 (5x)
        # open the image file with openslide for info and tifffile as zarr
        try:
            dask_array, x_res, y_res, _ = opensvs(svs_path, i)
            # units are natively ("nm", "nm") so no need to convert to get voxel size
            
            # convert to integer and calculate for each pyramid level
            voxel_size0 = Coordinate(int(x_res), int(y_res))
            expected_shape = tuple((s0_shape[0] // 2**i, s0_shape[1] // 2**i, 3)) # downsampled shape
            print(f"expected shape: {expected_shape}")
            print(f"actual shape: {dask_array.shape}")

            # check shape is expected shape
            if dask_array.shape == expected_shape:
                print("correct shape")
                
                # format data as funlib dataset
                raw = prepare_ds(
                    zarr_path / "raw" / f"s{i}",
                    dask_array.shape,
                    offset,
                    voxel_size,
                    axis_names,
                    units,
                    mode="w",
                    dtype=np.uint8,
                )
                
                # storage info
                store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
                dask_array = dask_array.rechunk(raw.data.chunksize)

                with ProgressBar():
                    dask.array.store(dask_array, store_rgb)

            else:
                voxel_size = tuple((voxel_size0[0] * 2**i, voxel_size0[0] * 2**i))
                # format data as funlib dataset
                raw = prepare_ds(
                    zarr_path / "raw" / f"s{i}",
                    expected_shape,
                    offset,
                    voxel_size,
                    axis_names,
                    units,
                    mode="w",
                    dtype=np.uint8,
                )
                # storage info
                store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
                prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
                print(f"chunk shape: {prev_layer.chunk_shape}")

                # mean downsampling
                factors = {0: 2, 1: 2}
                try:
                    dask_array = coarsen(mean, prev_layer.data, factors)
                except ValueError as e:
                    new_shape = tuple(
                        (
                            (prev_layer.data.shape[i] // factors[i]) * factors[i]
                            if i in factors
                            else prev_layer.data.shape[i]
                        )
                        for i in range(prev_layer.data.ndim)
                    )
                    dask_array_cropped = prev_layer.data[:new_shape[0], :new_shape[1], :new_shape[2]]
                    dask_array = coarsen(mean, dask_array_cropped, factors)
                # save to zarr
                with ProgressBar():
                    dask.array.store(dask_array, store_rgb)
        
        except TypeError as e: # if it finds an empty pyramid level, it fills it in
            print(f"for layer {i}: {e}")
            print("Generating layer")
            prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
            voxel_size = tuple((voxel_size0[0] * 2**i, voxel_size0[0] * 2**i))
            # format data as funlib dataset
            raw = prepare_ds(
                zarr_path / "raw" / f"s{i}",
                expected_shape,
                offset,
                voxel_size,
                axis_names,
                units,
                mode="w",
                dtype=np.uint8,
            )
            # storage info
            store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
            prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
            print(f"chunk shape: {prev_layer.chunk_shape}")
            # mean downsampling
            factors = {0: 2, 1: 2}
            try:
                dask_array = coarsen(mean, prev_layer.data, factors)
            except ValueError as e:
                new_shape = tuple(
                    (
                        (prev_layer.data.shape[i] // factors[i]) * factors[i]
                        if i in factors
                        else prev_layer.data.shape[i]
                    )
                    for i in range(prev_layer.data.ndim)
                )
                dask_array_cropped = prev_layer.data[:new_shape[0], :new_shape[1], :new_shape[2]]
                dask_array = coarsen(mean, dask_array_cropped, factors)
            # save to zarr
            with ProgressBar():
                dask.array.store(dask_array, store_rgb)
    return print("svs conversion complete")

In [ ]:
# --- convert .svs to .zarr
svs_path = Path(r"E:\[PROJ]_DCIS\BRACS\BRACS_1283.svs")
zarr_path = Path(r"E:\[PROJ]_DCIS\BRACS\BRACS_1283.zarr")
offset = Coordinate(0, 0)
axis_names = ['y', 'x', 'c^']

svs_to_zarr(svs_path, 
            zarr_path, 
            offset, 
            axis_names 
            )

### Claude Code Method

In [ ]:
# class name → final stored label value
CLASS_MAP = {
    "Normal": 0,
    "Benign": 1,
    "Malignant": 2,
    "FEA-sure": 3,
    "ADH-sure": 4,
    "DCIS-sure": 5,
}

def geojson_to_semantic_zarr_DEP(
    geojson_path,
    he_zarr_path,
    axis_names=("y", "x"),
    num_pyramid_levels=4,
    chunk_size=4096,
    class_priority=None,  # optional list of class name strings e.g. ["FEA-sure", "ADH-sure", "DCIS-sure"]
):
    """
    Convert QuPath GeoJSON annotations to a semantic label Zarr pyramid
    aligned to the H&E Zarr ("raw/s0") spatial metadata.

    INPUTS:
    - geojson_path (str): path to the GeoJSON file containing annotations
    - he_zarr_path (str): path to the H&E Zarr file (should contain "raw/s0" with correct spatial metadata)
    - axis_names (tuple): names of the spatial axes (default ("y", "x"))
    - num_pyramid_levels (int): number of pyramid levels present (default 4 for s0-s3)
    - chunk_size (int): size of chunks to process at a time when rasterizing (default 4096)
    - class_priority (list, optional): list of class NAME STRINGS in the order they should be drawn
      (earlier = lower priority, later = higher priority / wins overlaps)
      e.g. ["FEA-sure", "ADH-sure", "DCIS-sure"]

    OUTPUTS:
    - saves semantic label Zarr pyramid under "labels/s{level}" in the same directory as the H&E Zarr
    - label values in the output correspond exactly to CLASS_MAP values
    - unannotated/background pixels are stored as 255 (ignore label)
    """
    # ── read slide metadata from the H&E image Zarr ───────────────────────────
    img_ds0 = open_ds(he_zarr_path / "raw" / "s0")
    voxel_size0 = img_ds0.voxel_size
    offset0 = img_ds0.roi.get_begin()
    shape_yx = (img_ds0.roi.get_shape() / voxel_size0)
    height, width = int(shape_yx[0]), int(shape_yx[1])
    units = getattr(img_ds0, "units", None)

    # ── create s0 label store ─────────────────────────────────────────────────
    label_store0 = prepare_ds(
        he_zarr_path / "labels" / "s0",
        (height, width),
        offset0,
        voxel_size0,
        axis_names,
        units,
        mode="w",
        dtype=np.uint8,
    )

    # ── parse GeoJSON into per-class polygon lists ────────────────────────────
    with open(geojson_path) as f:
        geojson = json.load(f)

    polys_by_class = {}  # class_name (str) → list of shapely Polygons
    skipped_classes = set()  # track any class names not in CLASS_MAP

    for feature in geojson["features"]:
        class_name = feature["properties"]["classification"]["name"]

        # CHANGED: validate against CLASS_MAP immediately at parse time
        # so we catch unknown class names early with a clear warning
        if class_name not in CLASS_MAP:
            skipped_classes.add(class_name)
            continue

        geom_type = feature["geometry"]["type"]
        coords = feature["geometry"]["coordinates"]

        if geom_type == "Polygon":
            exterior = coords[0]
            holes = coords[1:]
            if len(exterior) < 3:
                continue
            poly = Polygon(exterior, holes)
            if not poly.is_valid:
                poly = poly.buffer(0)
            if poly.is_empty:
                continue
            polys_by_class.setdefault(class_name, []).append(poly)

# I dont know it includes MultiPolygon
        elif geom_type == "MultiPolygon":
            for poly_rings in coords:
                exterior = poly_rings[0]
                holes = poly_rings[1:]
                if len(exterior) < 3:
                    continue
                poly = Polygon(exterior, holes)
                if not poly.is_valid:
                    poly = poly.buffer(0)
                if poly.is_empty:
                    continue
                polys_by_class.setdefault(class_name, []).append(poly)

    if skipped_classes:
        print(f"WARNING: skipped features with unknown class names (not in CLASS_MAP): {skipped_classes}")

    # ── merge all polygons per class into one geometry ────────────────────────
    final_geom_by_class = {}  # class_name (str) → merged shapely geometry
    for class_name, polys in polys_by_class.items():
        geom = unary_union(polys)
        if not geom.is_empty:
            final_geom_by_class[class_name] = geom

    # ── draw order / priority ─────────────────────────────────────────────────
    if class_priority is None:
        # CHANGED: default sort is now by CLASS_MAP value (ascending) rather than
        # alphabetical by name, so lower-numbered classes are painted first
        # (= lower priority) and higher-numbered classes win overlaps.
        # This matches the intuition that higher CLASS_MAP values = more specific.
        draw_order = sorted(
            final_geom_by_class.keys(),
            key=lambda name: CLASS_MAP[name]
        )
    else:
        present = set(final_geom_by_class.keys())
        draw_order = [c for c in class_priority if c in present] + \
                     sorted(present - set(class_priority), key=lambda name: CLASS_MAP[name])

    # ── CHANGED: build paint_id lookup ───────────────────────────────────────
    # We cannot paint directly with CLASS_MAP values because CLASS_MAP["Normal"]=0
    # collides with the blank tile background (also 0). So during painting we use
    # a temporary internal ID = CLASS_MAP[name] + 1 (shifts everything up by 1,
    # keeping 0 free as the "nothing painted yet" sentinel).
    # At the end of each tile we remap back: paint_id → CLASS_MAP value, 0 → 255.
    paint_id = {name: CLASS_MAP[name] + 1 for name in draw_order}

    # print the final label mapping so the caller knows what values to expect
    print("Class → stored label mapping:")
    for name in draw_order:
        print(f"  '{name}' → {CLASS_MAP[name]}")
    print(f"  background/unannotated → 255 (ignore)")

    # ── helper: flatten Polygon / MultiPolygon ────────────────────────────────
    # (unchanged)
    def iter_polys(geom):
        if geom.is_empty:
            return
        gt = geom.geom_type
        if gt == "Polygon":
            yield geom
        elif gt == "MultiPolygon":
            yield from geom.geoms

    # ── helper: paint one polygon onto tile ───────────────────────────────────
    # (unchanged)
    def paint_polygon_with_holes(tile_mask, poly, class_id, y0, x0):
        ext = np.asarray(poly.exterior.coords)
        rr, cc = draw_polygon(ext[:, 1] - y0, ext[:, 0] - x0, tile_mask.shape)
        bg = (tile_mask[rr, cc] == 0)
        tile_mask[rr[bg], cc[bg]] = class_id
        for ring in poly.interiors:
            hole = np.asarray(ring.coords)
            rrh, cch = draw_polygon(hole[:, 1] - y0, hole[:, 0] - x0, tile_mask.shape)
            here = (tile_mask[rrh, cch] == class_id)
            tile_mask[rrh[here], cch[here]] = 0

    # ── rasterize s0 in chunks ────────────────────────────────────────────────
    for y0 in range(0, height, chunk_size):
        for x0 in range(0, width, chunk_size):
            y1 = min(y0 + chunk_size, height)
            x1 = min(x0 + chunk_size, width)
            tile_mask = np.zeros((y1 - y0, x1 - x0), dtype=np.uint8)
            tile_box = box(x0, y0, x1, y1)

            for class_name in draw_order:
                geom = final_geom_by_class[class_name]
                if not geom.intersects(tile_box):
                    continue
                clipped = geom.intersection(tile_box)
                if clipped.is_empty:
                    continue
                for poly in iter_polys(clipped):
                    paint_polygon_with_holes(tile_mask, poly, paint_id[class_name], y0, x0)

            # CHANGED: remap from temporary paint IDs back to CLASS_MAP values
            # paint_id[name] = CLASS_MAP[name] + 1, so we build an inverse lookup.
            # tile_mask == 0 means nothing was painted → store as 255 (ignore).
            IGNORE = np.uint8(255)
            remapped = np.full(tile_mask.shape, IGNORE, dtype=np.uint8)
            for class_name in draw_order:
                pid = paint_id[class_name]           # temporary paint value
                final_label = np.uint8(CLASS_MAP[class_name])  # true stored value
                remapped[tile_mask == pid] = final_label

            label_store0[y0:y1, x0:x1] = remapped

    print("labels/s0 written")

    # ── pyramid generation (max-pool downsample) ──────────────────────────────
    # (unchanged)
    prev = dask.array.from_zarr(he_zarr_path / "labels" / "s0")
    for level in range(1, num_pyramid_levels):
        voxel_size = voxel_size0 * (2**level)

        IGNORE = np.uint8(255)
        def reduce_block(x, axis=None):
            valid = x != IGNORE
            x0 = np.where(valid, x, 0).astype(np.uint8)
            m = x0.max(axis=axis)
            any_valid = valid.any(axis=axis)
            out = np.where(any_valid, m, IGNORE).astype(np.uint8)
            return out

        down = dask.array.coarsen(
            reduce_block,
            prev,
            {0: 2, 1: 2},
            trim_excess=True,
        )
        print(prev.shape, down.shape)
        print(down.dtype)

        level_store = prepare_ds(
            he_zarr_path / "labels" / f"s{level}",
            down.shape,
            offset0,
            voxel_size,
            axis_names,
            units,
            mode="w",
            dtype=np.uint8,
        )
        with ProgressBar():
            dask.array.store(down, level_store)
        prev = down
        print(f"labels/s{level} written")

    print("Semantic GeoJSON → Zarr conversion complete")

### My Code - Final

In [ ]:
# CLASS_MAP starts at 1, keeping 0 free as the blank tile background sentinel
CLASS_MAP = {
    "Benign-sure": 1,
    "Pathological-benign": 2,
    "UDH-sure": 3,
    "FEA-sure": 4,
    "ADH-sure": 5,
    "DCIS-sure": 6,
    "Malignant-sure":7,
}

def geojson_to_semantic_zarr(
    geojson_path,
    he_zarr_path,
    axis_names=("y", "x"),
    num_pyramid_levels=4,
    chunk_size=4096,
    class_priority=None,
):
    """
    Convert QuPath GeoJSON annotations to a semantic label Zarr pyramid
    aligned to the H&E Zarr ("raw/s0") spatial metadata.

    INPUTS:
    - geojson_path (str): path to the GeoJSON file containing annotations
    - he_zarr_path (str): path to the H&E Zarr file (should contain "raw/s0" with correct spatial metadata)
    - axis_names (tuple): names of the spatial axes (default ("y", "x"))
    - num_pyramid_levels (int): number of pyramid levels present (default 4 for s0-s3)
    - chunk_size (int): size of chunks to process at a time when rasterizing (default 4096)
    - class_priority (list, optional): list of class NAME STRINGS in the order they should be drawn
      (earlier = lower priority, later = higher priority / wins overlaps)
      e.g. ["FEA-sure", "ADH-sure", "DCIS-sure"]

    OUTPUTS:
    - saves semantic label Zarr pyramid under "labels/s{level}" in the same directory as the H&E Zarr
    - label values in the output are CLASS_MAP[name] - 1 (i.e. 1->0, 2->1, 3->2 ...)
    - unannotated/background pixels are stored as 255 (ignore label)
    """
    # read slide metadata from the h&e image zarr
    # NOTHING CHANGED
    img_ds0 = open_ds(he_zarr_path / "raw" / "s0")
    voxel_size0 = img_ds0.voxel_size
    offset0 = img_ds0.roi.get_begin()
    shape_yx = (img_ds0.roi.get_shape() / voxel_size0)
    height, width = int(shape_yx[0]), int(shape_yx[1])
    units = getattr(img_ds0, "units", None)
 
    # create s0 (40x magnification)
    # NOTHING CHANGED
    label_store0 = prepare_ds(
        he_zarr_path / "labels" / "s0",
        (height, width),
        offset0,
        voxel_size0,
        axis_names,
        units,
        mode="w",
        dtype=np.uint8,
    )

    # parse GeoJSON into per-class polygon lists
    with open(geojson_path) as f:
        geojson = json.load(f)

    polys_by_class = {}
    skipped_classes = set()

    for feature in geojson["features"]:
        # Identifying class name (FEA-sure, ADH-sure...)
        class_name = feature["properties"]["classification"]["name"]

        # There is one class named MALIGNANT, that I'm not sure belongs to which category.
        if class_name not in CLASS_MAP:
            skipped_classes.add(class_name)
            continue
        
        # Type = Polygon, dk why Multipolygon is added, might be able to remove that line.
        geom_type = feature["geometry"]["type"]
        coords = feature["geometry"]["coordinates"]

        if geom_type == "Polygon":
            # INCLUSION
            exterior = coords[0]
            # coords[1:] Should be empty for all the GeoJSON files
            # EXCLUSION
            holes = coords[1:]
            # make sure polygon is valid/exists
            # UNCHANGED
            if len(exterior) < 3:
                continue
            poly = Polygon(exterior, holes)
            if not poly.is_valid:
                poly = poly.buffer(0)
            if poly.is_empty:
                continue
            # Constructing Polygon
            polys_by_class.setdefault(class_name, []).append(poly)

        elif geom_type == "MultiPolygon":
            for poly_rings in coords:
                exterior = poly_rings[0]
                holes = poly_rings[1:]
                if len(exterior) < 3:
                    continue
                # Polygon constructors accepts both inclusion and exclusion polygons
                # Don't need to seperate them
                poly = Polygon(exterior, holes)
                if not poly.is_valid:
                    poly = poly.buffer(0)
                if poly.is_empty:
                    continue
                polys_by_class.setdefault(class_name, []).append(poly)

    # This will print when you encounter the images with 'MALIGNANT' as the class namme
    if skipped_classes:
        print(f"WARNING: skipped features with unknown class names (not in CLASS_MAP): {skipped_classes}")

    # merge all polygons per class into one geometry
    # UNCHANGED
    final_geom_by_class = {}
    for class_name, polys in polys_by_class.items():
        geom = unary_union(polys)
        if not geom.is_empty:
            final_geom_by_class[class_name] = geom

    # draw order / priority 
    # UNCHANGED
    if class_priority is None:
        draw_order = sorted(
            final_geom_by_class.keys(),
            key=lambda name: CLASS_MAP[name]
        )
    else:
        present = set(final_geom_by_class.keys())
        draw_order = [c for c in class_priority if c in present] + \
                     sorted(present - set(class_priority), key=lambda name: CLASS_MAP[name])

    # print the label mapping so the caller knows what values to expect
    print("Class → stored label mapping (after remapping):")
    for name in draw_order:
        print(f"  '{name}' (painted as {CLASS_MAP[name]}) → stored as {CLASS_MAP[name] - 1}")
    print(f"  background/unannotated → stored as 255 (ignore)")

    # UNCHANGED
    def iter_polys(geom):
        if geom.is_empty:
            return
        gt = geom.geom_type
        if gt == "Polygon":
            yield geom
        elif gt == "MultiPolygon":
            yield from geom.geoms

    # UNCHANGED
    def paint_polygon_with_holes(tile_mask, poly, class_id, y0, x0):
        """Paint polygons without overwriting other classes + correctly clear holes"""
        # paint exterior (background-only so we don't overwrite other classes)
        ext = np.asarray(poly.exterior.coords)
        rr, cc = draw_polygon(ext[:, 1] - y0, ext[:, 0] - x0, tile_mask.shape)
        bg = (tile_mask[rr, cc] == 0)
        tile_mask[rr[bg], cc[bg]] = class_id
        # clear holes, but only where we just painted this class
        for ring in poly.interiors:
            hole = np.asarray(ring.coords)
            rrh, cch = draw_polygon(hole[:, 1] - y0, hole[:, 0] - x0, tile_mask.shape)
            here = (tile_mask[rrh, cch] == class_id)
            tile_mask[rrh[here], cch[here]] = 0

    # rasterize s0 in chunks
    # UNCHANGED
    for y0 in range(0, height, chunk_size):
        for x0 in range(0, width, chunk_size):
            y1 = min(y0 + chunk_size, height)
            x1 = min(x0 + chunk_size, width)
            tile_mask = np.zeros((y1 - y0, x1 - x0), dtype=np.uint8)
            tile_box = box(x0, y0, x1, y1)

            for class_name in draw_order:
                geom = final_geom_by_class[class_name]
                class_id = CLASS_MAP[class_name]  # paint directly with CLASS_MAP value (1-indexed)
                if not geom.intersects(tile_box):
                    continue
                clipped = geom.intersection(tile_box)
                if clipped.is_empty:
                    continue
                for poly in iter_polys(clipped):
                    paint_polygon_with_holes(tile_mask, poly, class_id, y0, x0)

            # remap: 0 → 255 (ignore), 1 → 0, 2 → 1, 3 → 2 ...
            # identical to the XML pipeline remapping step
            IGNORE = 255
            remapped = np.full(tile_mask.shape, IGNORE, dtype=np.uint8)
            fg = tile_mask > 0
            remapped[fg] = tile_mask[fg] - 1
            label_store0[y0:y1, x0:x1] = remapped

    print("labels/s0 written")

    # pyramid generation (max-pool downsample) 
    # UNCHANGED
    prev = dask.array.from_zarr(he_zarr_path / "labels" / "s0")
    for level in range(1, num_pyramid_levels):
        voxel_size = voxel_size0 * (2**level)

        IGNORE = np.uint8(255)
        def reduce_block(x, axis=None):
            valid = x != IGNORE
            x0 = np.where(valid, x, 0).astype(np.uint8)
            m = x0.max(axis=axis)
            any_valid = valid.any(axis=axis)
            out = np.where(any_valid, m, IGNORE).astype(np.uint8)
            return out

        down = dask.array.coarsen(
            reduce_block,
            prev,
            {0: 2, 1: 2},
            trim_excess=True,
        )
        print(prev.shape, down.shape)
        print(down.dtype)

        level_store = prepare_ds(
            he_zarr_path / "labels" / f"s{level}",
            down.shape,
            offset0,
            voxel_size,
            axis_names,
            units,
            mode="w",
            dtype=np.uint8,
        )
        with ProgressBar():
            dask.array.store(down, level_store)
        prev = down
        print(f"labels/s{level} written")

    print("Semantic GeoJSON → Zarr conversion complete")

#### Single Image

In [4]:
# convert geojson to zarr...
geojson_path = Path(r"E:\[PROJ]_DCIS\BRACS\BRACS_1283.geojson")
he_zarr_path = Path(r"E:\[PROJ]_DCIS\BRACS\BRACS_1283_new_class.zarr")

# basic call with defaults
geojson_to_semantic_zarr(
    geojson_path=geojson_path,
    he_zarr_path=he_zarr_path,
)

Class → stored label mapping (after remapping):
  'FEA-sure' (painted as 4) → stored as 3
  'ADH-sure' (painted as 5) → stored as 4
  'DCIS-sure' (painted as 6) → stored as 5
  background/unannotated → stored as 255 (ignore)
labels/s0 written
(79857, 177288) (39928, 88644)
uint8
[########################################] | 100% Completed | 104.12 s
labels/s1 written
(39928, 88644) (19964, 44322)
uint8
[########################################] | 100% Completed | 65.15 s
labels/s2 written
(19964, 44322) (9982, 22161)
uint8
[########################################] | 100% Completed | 65.59 s
labels/s3 written
Semantic GeoJSON → Zarr conversion complete


#### Multiple Images

In [ ]:
slide_dir = Path(r"E:\PROJ_DCIS\annotations\annotation_scheme-1")
training_patches_dir = Path(r"E:\PROJ_DCIS\training_patches\annotation_scheme-1")

for slide in slide_dir.glob("*.svs"):
    slidename = slide.stem
    svs_path  = slide                          
    zarr_path = slide_dir / f"{slidename}.zarr"
    xml_path  = slide_dir / f"{slidename}.xml"
    output_dir = training_patches_dir / slidename
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'working on {slidename}!')

    # --- convert .svs to .zarr
    print('converting .svs to .zarr')
    offset = Coordinate(0, 0)
    axis_names = ['y', 'x', 'c^']
    svs_to_zarr(svs_path, 
                zarr_path, 
                offset, 
                axis_names
                )
    
    # --- convert .xml to .zarr
    print('converting .xml to .zarr')
    xml_to_semantic_zarr(xml_path,
                        zarr_path
                        )

    
